In [1]:
import torch
import torch.nn as nn
import torch.optim as optim
import numpy as np
import random
import numpy as np
import matplotlib.pyplot as plt
from sklearn.decomposition import PCA
from torch.optim.lr_scheduler import ReduceLROnPlateau
import seaborn as sns
import matplotlib.pyplot as plt


from utils import get_support_query_loaders, log_results_nn_ft, NeuralNetwork, save_best_model, classification_loss, propagation_loss, getPrAucIndividualClassFinetuning

In [2]:
data = np.load('./data/data_scaled.npz')
X_finetuning = data['X_finetuning']
y_ft1 = data['y_finetuning1']
y_ft2 = data['y_finetuning2']

In [3]:
y_ft2.shape

(18618, 4)

In [7]:
y_ft2[1:10]

array([[nan,  0., nan, nan],
       [nan, nan,  0., nan],
       [nan, nan,  0., nan],
       [ 0., nan, nan, nan],
       [nan, nan, nan,  0.],
       [nan, nan, nan, nan],
       [nan, nan, nan,  0.],
       [ 0., nan, nan, nan],
       [ 0., nan,  0., nan]])

In [11]:
num_tasks = y_ft2.shape[1]
counts = np.zeros((num_tasks, 3), dtype=int)  # cols: [#0, #1, #missing]

for i in range(num_tasks):
    col = y_ft2[:, i]
    counts[i, 0] = np.sum(col == 0)             # count zeros
    counts[i, 1] = np.sum(col == 1)             # count ones
    counts[i, 2] = np.sum(np.isnan(col))          # count None/missing

# print results
for i in range(num_tasks):
    zeros, ones, missing = counts[i]
    print(f"Task {i}: 0s = {zeros}, 1s = {ones}, missing = {missing}")

Task 0: 0s = 2919, 1s = 2, missing = 15697
Task 1: 0s = 2985, 1s = 6, missing = 15627
Task 2: 0s = 2939, 1s = 8, missing = 15671
Task 3: 0s = 2849, 1s = 3, missing = 15766
